# 04 — Comparativa Multi-Arquitectura (US-E3 / US-L1)

Este notebook carga múltiples experimentos de LangSmith y genera la tabla comparativa entre arquitecturas GraphRAG.

## Experimentos comparados:
| Wrapper | Descripción | Arquitectura |
|---|---|---|
| `graphrag_main` | validate_cypher=True | GraphRAG-LangChain |
| `graphrag_naive` | validate_cypher=False | GraphRAG-LangChain-Naive |
| `graphrag_no_context` | Sin contexto Neo4j | GraphRAG-NoContext |
| `graphrag_always_refuse` | Siempre rechaza | GraphRAG-AlwaysRefuse |
| `graphrag_langgraph` | LangGraph con retry | GraphRAG-LangGraph |

## Referencia académica:
- ARES (NAACL 2024): comparación calibrada entre arquitecturas
- RAGAS (EACL 2024): métricas de faithfulness + context


In [ ]:
import sys
sys.path.insert(0, '..')

from dotenv import load_dotenv
load_dotenv('../.env')

from langsmith import Client
import pandas as pd

client = Client()
print('LangSmith conectado ✅')

## 1. Ejecutar la tabla discriminativa (US-M1)

Si aún no has corrido los 4 wrappers, ejecuta desde terminal:
```bash
python scripts/discriminative_power.py
# o subset rápido:
python scripts/discriminative_power.py --subset 10
```

O desde este notebook:

In [ ]:
# Opción: correr solo always_refuse y no_context (los más rápidos, no requieren Neo4j)
from rag_eval.evaluators.universal import evaluate_rag_universal, DISCRIMINATIVE_EVALUATORS
from rag_eval.wrappers.graphrag_always_refuse import graphrag_always_refuse
from rag_eval.wrappers.graphrag_no_context import graphrag_no_context
from rag_eval.datasets.northwind import DATASET_NORTHWIND

DATASET = DATASET_NORTHWIND[:10]  # subset para test
DATASET_NAME = 'northwind-comparison-nb04'

print(f'Dataset: {len(DATASET)} ejemplos')

## 2. Cargar experimentos existentes de LangSmith

In [ ]:
# Listar experimentos disponibles
projects = list(client.list_projects())
print(f'Proyectos disponibles ({len(projects)}):')
for p in sorted(projects, key=lambda x: x.modified_at or '', reverse=True)[:20]:
    print(f'  {p.name}')

In [ ]:
# Configurar los nombres de los experimentos a comparar
# Ajusta estos nombres según los que aparezcan en la celda anterior
EXPERIMENT_NAMES = [
    # 'discriminative-graphrag_main-...',
    # 'discriminative-graphrag_naive-...',
    # 'discriminative-graphrag_no_context-...',
    # 'discriminative-graphrag_always_refuse-...',
]

# Si están vacíos, buscar automáticamente por prefijo
if not EXPERIMENT_NAMES:
    EXPERIMENT_NAMES = [
        p.name for p in projects 
        if 'discriminative' in p.name.lower()
    ]
    print(f'Experimentos encontrados automáticamente: {EXPERIMENT_NAMES}')

## 3. Generar tabla comparativa

In [ ]:
def load_experiment_as_dataframe(experiment_name: str) -> pd.DataFrame:
    """Descarga resultados de un experimento como DataFrame."""
    runs = list(client.list_runs(
        project_name=experiment_name,
        run_type='chain',
        limit=200,
    ))
    
    rows = []
    for run in runs:
        if run.feedback_stats:
            row = {'experiment': experiment_name}
            for key, stats in run.feedback_stats.items():
                score = stats.get('avg', stats.get('mean', None))
                if score is not None:
                    row[key] = score
            rows.append(row)
    
    return pd.DataFrame(rows)


all_dfs = []
for exp_name in EXPERIMENT_NAMES:
    df = load_experiment_as_dataframe(exp_name)
    if not df.empty:
        all_dfs.append(df)
        print(f'✅ {exp_name}: {len(df)} runs')
    else:
        print(f'⚠️  {exp_name}: sin datos')

if all_dfs:
    combined_df = pd.concat(all_dfs, ignore_index=True)
    print(f'\nTotal: {len(combined_df)} runs combinados')
else:
    print('No hay datos. Ejecuta primero scripts/discriminative_power.py')
    combined_df = pd.DataFrame()

In [ ]:
if not combined_df.empty:
    # Tabla de promedios por experimento
    KEY_METRICS = [
        'faithfulness_nli',
        'hallucination_rate', 
        'correctness_continuous',
        'negative_rejection',
    ]
    
    available_metrics = [m for m in KEY_METRICS if m in combined_df.columns]
    
    comparison_table = (
        combined_df
        .groupby('experiment')[available_metrics]
        .mean()
        .round(3)
    )
    
    print('\n📊 TABLA DE PODER DISCRIMINATIVO')
    print('=' * 60)
    display(comparison_table.style
        .format('{:.3f}')
        .background_gradient(cmap='RdYlGn', axis=0)
    )
else:
    print('Ejecuta primero scripts/discriminative_power.py para generar los experimentos')

## 4. Validación de poder discriminativo (criterios de aceptación US-G1 / US-G2)

In [ ]:
if not combined_df.empty:
    metrics_by_wrapper = comparison_table.to_dict('index')
    
    # US-G1: hallucination_rate ≥ 0.7 en no_context vs ≤ 0.3 en main
    no_ctx_key = next((k for k in metrics_by_wrapper if 'no_context' in k), None)
    main_key = next((k for k in metrics_by_wrapper if 'graphrag_main' in k or ('main' in k and 'no' not in k)), None)
    refuse_key = next((k for k in metrics_by_wrapper if 'always_refuse' in k), None)
    
    print('📋 CRITERIOS DE ACEPTACIÓN')
    print('─' * 50)
    
    if no_ctx_key and main_key and 'hallucination_rate' in combined_df.columns:
        hall_noctx = metrics_by_wrapper[no_ctx_key].get('hallucination_rate', None)
        hall_main = metrics_by_wrapper[main_key].get('hallucination_rate', None)
        if hall_noctx is not None and hall_main is not None:
            ok = hall_noctx >= 0.7 and hall_main <= 0.3
            print(f'US-G1 hallucination: no_context={hall_noctx:.3f} (≥0.7), main={hall_main:.3f} (≤0.3) → {"✅" if ok else "⚠️"}')
    
    if refuse_key:
        corr = metrics_by_wrapper[refuse_key].get('correctness_continuous', None)
        neg = metrics_by_wrapper[refuse_key].get('negative_rejection', None)
        if corr is not None:
            ok = corr <= 0.1
            print(f'US-G2 correctness:  always_refuse={corr:.3f} (≈0.0) → {"✅" if ok else "⚠️"}')
        if neg is not None:
            ok = neg >= 0.9
            print(f'US-G2 neg_rejection: always_refuse={neg:.3f} (≈1.0) → {"✅" if ok else "⚠️"}')

## 5. Confidence Score — comparativa calibrada

In [ ]:
# Si tenemos suficientes datos, entrenar el modelo logístico (US-C1)
if not combined_df.empty and len(combined_df) >= 10:
    from rag_eval.evaluators.universal import train_confidence_weights, compute_ece
    
    feature_keys = ['faithfulness_nli', 'hallucination_rate', 'correctness_continuous']
    available = [k for k in feature_keys if k in combined_df.columns]
    
    if 'correctness_continuous' in combined_df.columns and len(available) >= 2:
        metrics_list = combined_df[available].dropna().to_dict('records')
        labels = combined_df.loc[combined_df[available].notna().all(axis=1), 'correctness_continuous'].tolist()
        
        if len(metrics_list) >= 5:
            model = train_confidence_weights(metrics_list, labels, feature_keys=available)
            
            print('🔧 PESOS APRENDIDOS (US-C1)')
            print('─' * 40)
            for feat, w in model['weights'].items():
                bar = '█' * int(abs(w) * 20)
                print(f'  {feat:<35} {w:+.4f} {bar}')
            print(f'  ECE heurístico: (ver scripts/calibrate_confidence.py)')
            print(f'  ECE aprendido:  {model["ece_logistic"]:.4f}')
        else:
            print(f'⚠️  Solo {len(metrics_list)} ejemplos con features completos. Se necesitan ≥5.')
    else:
        print('⚠️  Faltan métricas para entrenar el modelo. Ejecuta con preset="default" o "full".')
else:
    print('No hay datos suficientes para calibración.')

## 6. Exportar tabla


In [ ]:
import os

if not combined_df.empty:
    os.makedirs('../results', exist_ok=True)
    combined_df.to_csv('../results/comparison_all.csv', index=False)
    if 'comparison_table' in dir():
        comparison_table.to_csv('../results/comparison_summary.csv')
    print('✅ Exportado a results/comparison_all.csv y results/comparison_summary.csv')
else:
    print('Sin datos para exportar.')